# Nano-U: 4-Fold Leave-One-Scene-Out (LOSO) Training on GPU
**Master's Thesis:** *Extending IoT-Robotics Convergence to Learned Perception*
**Author:** Tharen Candi (IAS-Lab / VIMP-Lab, University of Padua)

> **Hardware:** Ensure this runtime is set to **T4 GPU** ( →  → ).

In [ ]:
# 1. Environment & Repository Setup
!git clone https://github.com/tharencandi/Nano-U.git /content/Nano-U
%cd /content/Nano-U
!pip install -q tf-keras tensorflow-model-optimization pyyaml

import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", gpus)
assert len(gpus) > 0, "ERROR: Connect to a GPU runtime before proceeding!"

### 2. Dataset Setup
Mount your Google Drive to load  directly without waiting for rate-limited API downloads.

In [ ]:
# 2. Extract Dataset from Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile
zip_candidates = [
    "/content/drive/MyDrive/TinyAgri_full.zip",
    "/content/drive/MyDrive/thesis/TinyAgri_full.zip",
    "/content/TinyAgri_full.zip"
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)
assert zip_path is not None, f"Could not find TinyAgri_full.zip! Checked: {zip_candidates}"

print(f"Extracting {zip_path} into /content/Nano-U/data/...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/Nano-U/data/")
print("Extraction complete!")
!ls -la /content/Nano-U/data/TinyAgri_full/

In [ ]:
# 3. Build 4-Fold Leave-One-Scene-Out (LOSO) Splits
import re, json, yaml
from pathlib import Path

REPO_ROOT = Path("/content/Nano-U")
DATA_ROOT = REPO_ROOT / "data" / "TinyAgri_full"
MANIFEST_ROOT = REPO_ROOT / "data" / "manifests"
SPLITS_ROOT = REPO_ROOT / "data" / "cv_splits"
CONFIG_ROOT = REPO_ROOT / "config"
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
SPLITS_ROOT.mkdir(parents=True, exist_ok=True)

SCENE_NAMES = ["Crops_scene1", "Crops_scene2", "Tomatoes_scene1", "Tomatoes_scene2"]

def get_fnum(p):
    m = re.search(r"frame(\d+)\.png", p.name)
    return int(m.group(1)) if m else 0

scene_data = {}
for s in SCENE_NAMES:
    s_dir = DATA_ROOT / s
    imgs = sorted(list((s_dir / "img").glob("*.png")), key=get_fnum)
    masks = sorted(list((s_dir / "mask").glob("*.png")), key=get_fnum)
    pairs = [{"img": str(ip.resolve()), "mask": str(mp.resolve()), "filename": ip.name, "scene": s, "frame": get_fnum(ip)} for ip, mp in zip(imgs, masks)]
    scene_data[s] = pairs
    print(f"Loaded {s}: {len(pairs)} frames")

for fold_idx, test_s in enumerate(SCENE_NAMES):
    fold_dir = SPLITS_ROOT / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    manifest_dir = MANIFEST_ROOT / f"cv_fold_{fold_idx}"
    manifest_dir.mkdir(parents=True, exist_ok=True)

    train_scenes = [s for s in SCENE_NAMES if s != test_s]
    train_pairs = []
    val_pairs = []
    for ts in train_scenes:
        plist = scene_data[ts]
        n = len(plist)
        n_val = max(1, round(n * 0.15))
        val_start = n - n_val
        train_end = max(1, val_start - 20) # 20-frame temporal buffer gap
        train_pairs.extend(plist[:train_end])
        val_pairs.extend(plist[val_start:])

    test_pairs = list(scene_data[test_s])

    # Manifests
    with open(manifest_dir / "train.json", "w") as f: json.dump(train_pairs, f, indent=2)
    with open(manifest_dir / "val.json", "w") as f: json.dump(val_pairs, f, indent=2)
    with open(manifest_dir / "test.json", "w") as f: json.dump(test_pairs, f, indent=2)

    # Folder symlinks
    for split_name, spairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
        s_img, s_mask = fold_dir / split_name / "img", fold_dir / split_name / "mask"
        s_img.mkdir(parents=True, exist_ok=True)
        s_mask.mkdir(parents=True, exist_ok=True)
        for p in spairs:
            dst_i, dst_m = s_img / p["filename"], s_mask / p["filename"]
            if not dst_i.exists(): dst_i.symlink_to(Path(p["img"]))
            if not dst_m.exists(): dst_m.symlink_to(Path(p["mask"]))

    # Config YAML
    with open(CONFIG_ROOT / "TinyAgri_config.yaml") as f: cfg = yaml.safe_load(f)
    fold_split_rel = f"data/cv_splits/fold_{fold_idx}"
    cfg["data"]["paths"]["processed"]["root"] = fold_split_rel
    cfg["data"]["paths"]["processed"]["train"]["img"] = f"{fold_split_rel}/train/img"
    cfg["data"]["paths"]["processed"]["train"]["mask"] = f"{fold_split_rel}/train/mask"
    cfg["data"]["paths"]["processed"]["val"]["img"] = f"{fold_split_rel}/val/img"
    cfg["data"]["paths"]["processed"]["val"]["mask"] = f"{fold_split_rel}/val/mask"
    cfg["data"]["paths"]["processed"]["test"]["img"] = f"{fold_split_rel}/test/img"
    cfg["data"]["paths"]["processed"]["test"]["mask"] = f"{fold_split_rel}/test/mask"
    cfg["data"]["paths"]["models_dir"] = f"models/TinyAgri_fold{fold_idx}"
    cfg["data"]["paths"]["results_dir"] = f"results/TinyAgri_fold{fold_idx}"
    cfg["training"]["bu_net"]["models_dir"] = f"models/TinyAgri_fold{fold_idx}"
    cfg["training"]["nano_u"]["models_dir"] = f"models/TinyAgri_fold{fold_idx}"
    cfg["training"]["nano_u"]["distillation"]["teacher_weights"] = f"models/TinyAgri_fold{fold_idx}/bu_net.h5"

    out_cfg = CONFIG_ROOT / f"TinyAgri_fold{fold_idx}.yaml"
    with open(out_cfg, "w") as f: yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    print(f"Fold {fold_idx}: Test={test_s:<16} | Train={len(train_pairs):4d} | Val={len(val_pairs):3d} | Test={len(test_pairs):4d}")

print("
All 4 folds initialized!")

In [ ]:
# 4. Patch evaluate.py for Foreground & Background IoU
eval_file = Path("/content/Nano-U/src/evaluate.py")
content = eval_file.read_text()
if ""iou_foreground"" not in content:
    content = content.replace(
        "return {\n",
        "return {\n        "iou_foreground": float(iou_fg),\n        "iou_background": float(iou_bg),\n"
    )
    eval_file.write_text(content)
    print("Patched src/evaluate.py!")

In [ ]:
# 5. Execute 4-Fold QAD Training Loop on GPU
import subprocess

for fold_idx in range(4):
    cfg = f"config/TinyAgri_fold{fold_idx}.yaml"
    print(f"\n{"="*50}\n  STARTING FOLD {fold_idx} ({cfg})\n{"="*50}")
    ret = subprocess.run(["python", "scripts/run_qad.py", "--config", cfg])
    if ret.returncode != 0:
        raise RuntimeError(f"Fold {fold_idx} failed!")
    print(f"Fold {fold_idx} finished successfully!")

In [ ]:
# 6. Aggregate Summary and Package for Download
import os, json, subprocess
from google.colab import files

cv_summary = {}
for i in range(4):
    p = f"results/TinyAgri_fold{i}/pipeline_summary.json"
    if os.path.exists(p):
        with open(p) as f: cv_summary[f"fold_{i}"] = json.load(f)

with open("results/cv_4folds_summary.json", "w") as f:
    json.dump(cv_summary, f, indent=2)
print("Saved results/cv_4folds_summary.json!")

print("Packaging results into nanou_4folds_trained.zip...")
!zip -r -q nanou_4folds_trained.zip models results config
print("Ready for download!")
files.download("nanou_4folds_trained.zip")